In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os


Mounted at /content/drive


In [3]:

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
import numpy as np

# Dataset class to handle email text data and labels
class EmailDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts  # Preprocessed text data
        self.labels = labels  # Corresponding labels

    def __len__(self):
        return len(self.texts) # Return total number of samples

    def __getitem__(self, idx):
      # Return a single sample and its label as tensors
        return torch.tensor(self.texts[idx], dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.long)

# LSTM-based classifier model for email classification
class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)  # LSTM layer
        self.fc = nn.Linear(hidden_dim, output_dim) # Fully connected layer for classification

    def forward(self, x):
        # Forward pass through the LSTM and FC layers
        _, (hidden, _) = self.lstm(x) # Use the final hidden state
        out = self.fc(hidden[-1])  # Pass through the fully connected layer
        return out

# Load dataset
file_path = '/content/drive/MyDrive/emails.xlsx'
data = pd.read_excel(file_path)# Load email data from an Excel file

# Preprocessing text data
vectorizer = CountVectorizer(max_features=5000)# Convert text to a bag-of-words representation
X = vectorizer.fit_transform(data['text']).toarray() # Transform text data into numerical format
label_encoder = LabelEncoder() # Encode labels (e.g., spam or not spam)
y = label_encoder.fit_transform(data['spam'])

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert data into PyTorch Dataset objects
train_dataset = EmailDataset(X_train, y_train)  # Training dataset
test_dataset = EmailDataset(X_test, y_test) # Test dataset
# Create DataLoader objects for batch processing
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True) # Shuffle training data
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False) # No need to shuffle test data

# Model Parameters
input_dim = 5000
hidden_dim = 128 # Number of units in LSTM hidden layer
output_dim = len(label_encoder.classes_)
num_layers = 2
epochs = 10
learning_rate = 0.001

# Initialize model, loss, and optimizer
model = LSTMClassifier(input_dim, hidden_dim, output_dim, num_layers)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
for epoch in range(epochs):
    model.train() # Set the model to training mode
    total_loss = 0  # Track total loss for the epoch
    for texts, labels in train_loader:
        texts = texts.unsqueeze(1)  # Add sequence dimension
        optimizer.zero_grad()
        outputs = model(texts) # Forward pass
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

# Evaluation
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for texts, labels in test_loader:
        texts = texts.unsqueeze(1)
        outputs = model(texts)
        _, predicted = torch.max(outputs, 1)
        y_true.extend(labels.tolist())
        y_pred.extend(predicted.tolist())

# Metrics
conf_matrix = confusion_matrix(y_true, y_pred)
accuracy = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='weighted')

print("Confusion Matrix:")
print(conf_matrix)
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")

# Save model
model_path = '/content/drive/My Drive/lstm_email_classifier.pth'
torch.save(model.state_dict(), model_path)
print(f"Model saved to {model_path}")

# Load and test saved model
loaded_model = LSTMClassifier(input_dim, hidden_dim, output_dim, num_layers)
loaded_model.load_state_dict(torch.load(model_path))
loaded_model.eval()

# Evaluate loaded model
y_loaded_pred = []
with torch.no_grad():
    for texts, labels in test_loader:
        texts = texts.unsqueeze(1)
        outputs = loaded_model(texts)
        _, predicted = torch.max(outputs, 1)
        y_loaded_pred.extend(predicted.tolist())

loaded_accuracy = accuracy_score(y_true, y_loaded_pred)
print(f"Loaded Model Accuracy: {loaded_accuracy:.4f}")



Epoch 1/10, Loss: 23.5823
Epoch 2/10, Loss: 1.3179
Epoch 3/10, Loss: 0.3128
Epoch 4/10, Loss: 0.1522
Epoch 5/10, Loss: 0.0940
Epoch 6/10, Loss: 0.0641
Epoch 7/10, Loss: 0.0494
Epoch 8/10, Loss: 0.0371
Epoch 9/10, Loss: 0.0299
Epoch 10/10, Loss: 0.0239
Confusion Matrix:
[[852   4]
 [  4 286]]
Accuracy: 0.9930
F1 Score: 0.9930
Model saved to /content/drive/My Drive/lstm_email_classifier.pth


<ipython-input-3-60272c1ee44c>:113: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loaded_model.load_state_dict(torch.load(model_path))


Loaded Model Accuracy: 0.9930
